# EPIC Clarity Visit Detail Hydration

This notebook hydrates the OMOP VISIT_DETAIL table from EPIC Clarity ADT (Admit/Discharge/Transfer) events.

## Source Tables
- `_exponent._bronze_epic_clarity_*.dbo_CLARITY_ADT` - Admission/discharge/transfer events
- `_exponent._bronze_epic_clarity_*.dbo_PAT_ENC_HSP` - Hospital encounter with patient class information

## OMOP Fields Populated
- visit_detail_id (surrogate key)
- visit_detail_source_value
- visit_detail_concept_id (mapped from event type/patient class)
- visit_detail_start_datetime (ADT effective time)
- visit_occurrence_id (parent encounter)
- care_site_id (bed/unit location if available)

## Notes
- VISIT_DETAIL represents intra-hospital movements
- Captures ADT events (admission, discharge, transfer)
- Links to parent visit_occurrence for the overall hospital stay
- Useful for tracking patient movement across units/beds

In [ ]:
source = 'epic_clarity'

In [ ]:
silver_visit_detail_df = spark.sql("""
WITH visit_detail_with_duplicates AS (
  SELECT
    pe.person_id,
    0 AS visit_detail_concept_id,
    CAST(ca.EFFECTIVE_TIME AS DATE) AS visit_detail_start_date,
    ca.EFFECTIVE_TIME AS visit_detail_start_datetime,
    CAST(ca.EFFECTIVE_TIME AS DATE) AS visit_detail_end_date,
    ca.EFFECTIVE_TIME AS visit_detail_end_datetime,
    0 AS visit_detail_type_concept_id,
    NULL AS provider_id,
    NULL AS care_site_id,
    CONCAT_WS(CHR(31), 'epic_clarity', 'CLARITY_ADT', 'EVENT_ID', CAST(ca.EVENT_ID AS STRING)) AS visit_detail_source_value,
    0 AS visit_detail_source_concept_id,
    0 AS admitted_from_concept_id,
    NULL AS admitted_from_source_value,
    0 AS discharged_to_concept_id,
    NULL AS discharged_to_source_value,
    NULL AS preceding_visit_detail_id,
    NULL AS parent_visit_detail_id,
    vo.visit_occurrence_id,
    'epic_clarity' AS source_system,
    CURRENT_TIMESTAMP() AS last_mod_tsp,
    ROW_NUMBER() OVER (PARTITION BY CONCAT_WS(CHR(31), 'epic_clarity', 'CLARITY_ADT', 'EVENT_ID', CAST(ca.EVENT_ID AS STRING)) ORDER BY ca.EFFECTIVE_TIME DESC) as rn
  FROM _exponent._bronze_epic_clarity.clarity_adt ca
  INNER JOIN _exponent._bronze_epic_clarity.pat_enc enc
    ON ca.PAT_ENC_CSN_ID = enc.PAT_ENC_CSN_ID
  INNER JOIN _exponent.omop_mapping.source_to_person pe
    ON CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', CAST(enc.PAT_ID AS STRING)) = pe.person_source_value
    AND pe.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence vo
    ON CONCAT_WS(CHR(31), 'epic_clarity', 'PAT_ENC', 'PAT_ENC_CSN_ID', CAST(ca.PAT_ENC_CSN_ID AS STRING)) = vo.visit_occurrence_source_value
    AND vo.active_flag = TRUE
  WHERE ca.EVENT_ID IS NOT NULL
    AND ca.EFFECTIVE_TIME IS NOT NULL
)
SELECT
  person_id,
  visit_detail_concept_id,
  visit_detail_start_date,
  visit_detail_start_datetime,
  visit_detail_end_date,
  visit_detail_end_datetime,
  visit_detail_type_concept_id,
  provider_id,
  care_site_id,
  visit_detail_source_value,
  visit_detail_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_detail_id,
  parent_visit_detail_id,
  visit_occurrence_id,
  source_system,
  last_mod_tsp
FROM visit_detail_with_duplicates
WHERE rn = 1
""")

# display(silver_visit_detail_df)
silver_visit_detail_df.createOrReplaceTempView("silver_visit_detail")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.visit_detail AS target
USING silver_visit_detail AS source
ON target.visit_detail_source_value = source.visit_detail_source_value

WHEN MATCHED AND NOT (
    target.person_id <=> source.person_id
    AND target.visit_detail_start_datetime <=> source.visit_detail_start_datetime
    AND target.visit_occurrence_id <=> source.visit_occurrence_id
)
THEN UPDATE SET
    target.person_id = source.person_id,
    target.visit_detail_concept_id = source.visit_detail_concept_id,
    target.visit_detail_start_date = source.visit_detail_start_date,
    target.visit_detail_start_datetime = source.visit_detail_start_datetime,
    target.visit_detail_end_date = source.visit_detail_end_date,
    target.visit_detail_end_datetime = source.visit_detail_end_datetime,
    target.visit_detail_type_concept_id = source.visit_detail_type_concept_id,
    target.provider_id = source.provider_id,
    target.care_site_id = source.care_site_id,
    target.visit_detail_source_concept_id = source.visit_detail_source_concept_id,
    target.admitted_from_concept_id = source.admitted_from_concept_id,
    target.admitted_from_source_value = source.admitted_from_source_value,
    target.discharged_to_concept_id = source.discharged_to_concept_id,
    target.discharged_to_source_value = source.discharged_to_source_value,
    target.preceding_visit_detail_id = source.preceding_visit_detail_id,
    target.parent_visit_detail_id = source.parent_visit_detail_id,
    target.visit_occurrence_id = source.visit_occurrence_id,
    target.last_mod_tsp = source.last_mod_tsp

WHEN NOT MATCHED THEN INSERT (
    person_id,
    visit_detail_concept_id,
    visit_detail_start_date,
    visit_detail_start_datetime,
    visit_detail_end_date,
    visit_detail_end_datetime,
    visit_detail_type_concept_id,
    provider_id,
    care_site_id,
    visit_detail_source_value,
    visit_detail_source_concept_id,
    admitted_from_concept_id,
    admitted_from_source_value,
    discharged_to_concept_id,
    discharged_to_source_value,
    preceding_visit_detail_id,
    parent_visit_detail_id,
    visit_occurrence_id,
    source_system,
    last_mod_tsp
)
VALUES (
    source.person_id,
    source.visit_detail_concept_id,
    source.visit_detail_start_date,
    source.visit_detail_start_datetime,
    source.visit_detail_end_date,
    source.visit_detail_end_datetime,
    source.visit_detail_type_concept_id,
    source.provider_id,
    source.care_site_id,
    source.visit_detail_source_value,
    source.visit_detail_source_concept_id,
    source.admitted_from_concept_id,
    source.admitted_from_source_value,
    source.discharged_to_concept_id,
    source.discharged_to_source_value,
    source.preceding_visit_detail_id,
    source.parent_visit_detail_id,
    source.visit_occurrence_id,
    source.source_system,
    source.last_mod_tsp
)

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_visit_detail (
    visit_detail_source_value,
    active_flag,
    created_tsp
)
SELECT
    visit_detail_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp
FROM _exponent.omop_silver.visit_detail
WHERE visit_detail_source_value IS NOT NULL
  AND source_system = 'epic_clarity'
  AND visit_detail_source_value NOT IN (
    SELECT visit_detail_source_value
    FROM _exponent.omop_mapping.source_to_visit_detail
    WHERE active_flag = TRUE
  )

In [ ]:
gold_visit_detail_df = spark.sql("""
WITH gold_with_duplicates AS (
  SELECT
    m.visit_detail_id,
    s.person_id,
    s.visit_occurrence_id,
    s.visit_detail_concept_id,
    s.visit_detail_start_date,
    s.visit_detail_start_datetime,
    s.visit_detail_end_date,
    s.visit_detail_end_datetime,
    s.visit_detail_type_concept_id,
    s.provider_id,
    s.care_site_id,
    s.visit_detail_source_concept_id,
    s.admitted_from_concept_id,
    s.admitted_from_source_value,
    s.discharged_to_concept_id,
    s.discharged_to_source_value,
    s.preceding_visit_detail_id,
    s.parent_visit_detail_id,
    ROW_NUMBER() OVER (PARTITION BY m.visit_detail_id ORDER BY s.last_mod_tsp DESC) as rn
  FROM _exponent.omop_silver.visit_detail s
  INNER JOIN _exponent.omop_mapping.source_to_visit_detail m
    ON s.visit_detail_source_value = m.visit_detail_source_value
    AND m.active_flag = TRUE
  WHERE s.source_system = 'epic_clarity'
)
SELECT
  visit_detail_id,
  person_id,
  visit_occurrence_id,
  visit_detail_concept_id,
  visit_detail_start_date,
  visit_detail_start_datetime,
  visit_detail_end_date,
  visit_detail_end_datetime,
  visit_detail_type_concept_id,
  provider_id,
  care_site_id,
  visit_detail_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_detail_id,
  parent_visit_detail_id
FROM gold_with_duplicates
WHERE rn = 1
""")

# display(gold_visit_detail_df)
gold_visit_detail_df.createOrReplaceTempView("gold_visit_detail")

In [ ]:
%sql
MERGE INTO _exponent.omop.visit_detail AS target
USING gold_visit_detail AS source
ON target.visit_detail_id = source.visit_detail_id

WHEN MATCHED AND NOT (
    target.person_id <=> source.person_id
    AND target.visit_occurrence_id <=> source.visit_occurrence_id
    AND target.visit_detail_concept_id <=> source.visit_detail_concept_id
    AND target.visit_detail_start_date <=> source.visit_detail_start_date
    AND target.visit_detail_start_datetime <=> source.visit_detail_start_datetime
)
THEN UPDATE SET
    target.person_id = source.person_id,
    target.visit_occurrence_id = source.visit_occurrence_id,
    target.visit_detail_concept_id = source.visit_detail_concept_id,
    target.visit_detail_start_date = source.visit_detail_start_date,
    target.visit_detail_start_datetime = source.visit_detail_start_datetime,
    target.visit_detail_end_date = source.visit_detail_end_date,
    target.visit_detail_end_datetime = source.visit_detail_end_datetime,
    target.visit_detail_type_concept_id = source.visit_detail_type_concept_id,
    target.provider_id = source.provider_id,
    target.care_site_id = source.care_site_id,
    target.visit_detail_source_concept_id = source.visit_detail_source_concept_id,
    target.admitted_from_concept_id = source.admitted_from_concept_id,
    target.admitted_from_source_value = source.admitted_from_source_value,
    target.discharged_to_concept_id = source.discharged_to_concept_id,
    target.discharged_to_source_value = source.discharged_to_source_value,
    target.preceding_visit_detail_id = source.preceding_visit_detail_id,
    target.parent_visit_detail_id = source.parent_visit_detail_id

WHEN NOT MATCHED THEN INSERT (
    visit_detail_id,
    person_id,
    visit_occurrence_id,
    visit_detail_concept_id,
    visit_detail_start_date,
    visit_detail_start_datetime,
    visit_detail_end_date,
    visit_detail_end_datetime,
    visit_detail_type_concept_id,
    provider_id,
    care_site_id,
    visit_detail_source_concept_id,
    admitted_from_concept_id,
    admitted_from_source_value,
    discharged_to_concept_id,
    discharged_to_source_value,
    preceding_visit_detail_id,
    parent_visit_detail_id
)
VALUES (
    source.visit_detail_id,
    source.person_id,
    source.visit_occurrence_id,
    source.visit_detail_concept_id,
    source.visit_detail_start_date,
    source.visit_detail_start_datetime,
    source.visit_detail_end_date,
    source.visit_detail_end_datetime,
    source.visit_detail_type_concept_id,
    source.provider_id,
    source.care_site_id,
    source.visit_detail_source_concept_id,
    source.admitted_from_concept_id,
    source.admitted_from_source_value,
    source.discharged_to_concept_id,
    source.discharged_to_source_value,
    source.preceding_visit_detail_id,
    source.parent_visit_detail_id
)